# Current Portfolio Report Preview

This notebook is the working portfolio-monitor checkpoint for the new universe-first stack. It does three things:

1. Sync the generated `configs/portfolio_monitor/security_reference.csv` from `configs/security_universe.csv`.
2. Build a current position report either from live TWS or by rebuilding from an existing live-report CSV.
3. Show the report pieces we can already generate today: top positions, asset-class exposure, EQ country, US sector, and FI tenor breakdowns.

Recommended kernel: `py313`.


In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
while not (project_root / "market_helper").exists():
    if project_root.parent == project_root:
        raise RuntimeError("Could not find project root containing the market_helper package.")
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

def load_local_account_defaults(root: Path) -> dict[str, str]:
    candidates = [
        root / "configs" / "portfolio_monitor" / "report_accounts.local.env",
        root / "configs" / "report_accounts.local.env",
    ]
    values: dict[str, str] = {}
    for path in candidates:
        if not path.exists():
            continue
        for line in path.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            values[key.strip()] = value.strip().strip(chr(34)).strip("'")
        break
    return values

local_account_defaults = load_local_account_defaults(project_root)

# Recommended default for reliable notebook replay: rebuild from the existing live CSV.
# Switch SOURCE_MODE to "live" when TWS / IB Gateway is running locally.
SOURCE_MODE = "rebuild_from_existing_csv"  # "live" or "rebuild_from_existing_csv"
EXISTING_LIVE_CSV = project_root / "outputs" / "reports" / "live_ibkr_position_report.csv"
POSITION_REPORT_OUTPUT = project_root / "outputs" / "reports" / "current_portfolio_report_preview.csv"
RISK_HTML_OUTPUT = project_root / "outputs" / "reports" / "current_portfolio_risk_preview.html"
SECURITY_REFERENCE_OUTPUT = project_root / "configs" / "portfolio_monitor" / "security_reference.csv"

TRY_BUILD_RISK_HTML = False
RETURNS_JSON = None
PROXY_JSON = None
REGIME_JSON = None

IB_HOST = "127.0.0.1"
IB_PORT = 7497
IB_CLIENT_ID = 123
IB_ACCOUNT = local_account_defaults.get("DEFAULT_PROD_ACCOUNT_ID", "")

print(f"Project root: {project_root}")
print(f"Source mode: {SOURCE_MODE}")
print(f"Existing live CSV: {EXISTING_LIVE_CSV}")
print(f"Position report output: {POSITION_REPORT_OUTPUT}")
print(f"Generated security reference: {SECURITY_REFERENCE_OUTPUT}")
print(f"Default live account: {IB_ACCOUNT or '<unset>'}")


In [ ]:
from collections import defaultdict
import csv
import json
from zipfile import ZipFile
import xml.etree.ElementTree as ET

from IPython.display import HTML, Markdown, display
import pandas as pd

from market_helper.data_sources.ibkr.adapters import (
    normalize_ibkr_latest_prices,
    normalize_ibkr_positions,
)
from market_helper.portfolio import build_price_lookup
from market_helper.portfolio.security_reference import build_security_reference_table
from market_helper.presentation.exporters.csv import export_position_report_csv
from market_helper.presentation.tables.portfolio_report import build_position_report_rows
from market_helper.reporting.risk_html import (
    _build_eq_country_breakdown,
    _build_fi_tenor_breakdown,
    _build_us_sector_breakdown,
    _funded_aum,
    load_position_rows,
)
from market_helper.workflows.generate_report import (
    generate_live_ibkr_position_report,
    generate_risk_html_report,
    generate_security_reference_sync,
)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)

FUTURE_EXCHANGES = {"CBOT", "CFE", "CME", "COMEX", "ICE", "NYMEX"}

def infer_sec_type_from_row(row: dict[str, str]) -> str:
    exchange = str(row.get("exchange") or "").upper()
    symbol = str(row.get("symbol") or "").upper()
    if symbol.endswith("_CASH_VALUE"):
        return "CASH"
    if exchange in FUTURE_EXCHANGES:
        return "FUT"
    return "STK"

def rebuild_position_report_from_existing_live_csv(
    existing_csv_path: Path,
    output_path: Path,
) -> tuple[Path, object]:
    rows = list(csv.DictReader(existing_csv_path.open("r", encoding="utf-8", newline="")))
    if not rows:
        raise ValueError(f"No rows found in {existing_csv_path}")

    as_of = str(rows[0].get("as_of") or "") or None
    raw_positions: list[dict[str, str]] = []
    raw_prices: list[dict[str, str]] = []
    skipped_rows = 0

    for row in rows:
        con_id = str(row.get("con_id") or "").strip()
        sec_type = infer_sec_type_from_row(row)
        if not con_id and sec_type != "CASH":
            skipped_rows += 1
            continue
        raw_positions.append({
            "account": str(row.get("account") or ""),
            "con_id": con_id,
            "sec_type": sec_type,
            "symbol": str(row.get("symbol") or ""),
            "currency": str(row.get("currency") or ""),
            "exchange": str(row.get("exchange") or ""),
            "local_symbol": str(row.get("local_symbol") or ""),
            "position": str(row.get("quantity") or "0"),
            "avg_cost": str(row.get("avg_cost") or ""),
            "market_value": str(row.get("market_value") or ""),
        })
        if con_id and str(row.get("latest_price") or "").strip():
            raw_prices.append({
                "con_id": con_id,
                "marketPrice": str(row.get("latest_price") or ""),
            })

    reference_table = build_security_reference_table(reference_path=SECURITY_REFERENCE_OUTPUT)
    positions = normalize_ibkr_positions(raw_positions, reference_table, as_of=as_of)
    prices = normalize_ibkr_latest_prices(raw_prices, reference_table, as_of=as_of)
    report_rows = build_position_report_rows(
        positions,
        build_price_lookup(prices),
        reference_table.to_security_lookup(),
    )
    written_path = export_position_report_csv(report_rows, output_path)
    print(
        f"Rebuilt {len(report_rows)} normalized report rows from existing CSV "
        f"({skipped_rows} skipped rows without usable contract ids)."
    )
    return written_path, reference_table

def breakdown_df(rows: list[object]) -> pd.DataFrame:
    return pd.DataFrame([row.__dict__ for row in rows])

def preview_target_workbook_sheet(workbook_path: Path, sheet_name: str, max_rows: int = 10, max_cols: int = 14) -> pd.DataFrame:
    ns = {
        "a": "http://schemas.openxmlformats.org/spreadsheetml/2006/main",
        "r": "http://schemas.openxmlformats.org/officeDocument/2006/relationships",
        "pr": "http://schemas.openxmlformats.org/package/2006/relationships",
    }
    with ZipFile(workbook_path) as archive:
        workbook = ET.fromstring(archive.read("xl/workbook.xml"))
        rels = ET.fromstring(archive.read("xl/_rels/workbook.xml.rels"))
        relmap = {
            rel.attrib["Id"]: rel.attrib["Target"]
            for rel in rels.findall("pr:Relationship", ns)
        }
        shared_strings: list[str] = []
        if "xl/sharedStrings.xml" in archive.namelist():
            shared_root = ET.fromstring(archive.read("xl/sharedStrings.xml"))
            shared_strings = [
                "".join(text.text or "" for text in item.iterfind(".//a:t", ns))
                for item in shared_root.findall("a:si", ns)
            ]
        sheets = workbook.find("a:sheets", ns)
        matching = [sheet for sheet in sheets if sheet.attrib.get("name") == sheet_name]
        if not matching:
            raise KeyError(f"Workbook sheet not found: {sheet_name}")
        rid = matching[0].attrib["{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id"]
        target = relmap[rid]
        if not target.startswith("xl/"):
            target = f"xl/{target}"
        root = ET.fromstring(archive.read(target))
        materialized: list[list[object]] = []
        for row in root.findall(".//a:sheetData/a:row", ns)[:max_rows]:
            values: list[object] = []
            for cell in row.findall("a:c", ns)[:max_cols]:
                value = cell.find("a:v", ns)
                if value is None:
                    values.append(None)
                    continue
                if cell.attrib.get("t") == "s":
                    idx = int(value.text)
                    values.append(shared_strings[idx] if idx < len(shared_strings) else value.text)
                else:
                    values.append(value.text)
            materialized.append(values)
        return pd.DataFrame(materialized)


In [ ]:
synced_reference_path = generate_security_reference_sync(output_path=SECURITY_REFERENCE_OUTPUT)
print(f"Synced generated security reference to: {synced_reference_path}")

if SOURCE_MODE == "live":
    position_report_path = generate_live_ibkr_position_report(
        output_path=POSITION_REPORT_OUTPUT,
        host=IB_HOST,
        port=IB_PORT,
        client_id=IB_CLIENT_ID,
        account_id=IB_ACCOUNT or None,
        timeout=4.0,
    )
    reference_table = build_security_reference_table(reference_path=synced_reference_path)
    acquisition_note = "Generated directly from live TWS / IB Gateway."
else:
    if not EXISTING_LIVE_CSV.exists():
        raise FileNotFoundError(
            f"Fallback CSV not found at {EXISTING_LIVE_CSV}. Either switch SOURCE_MODE to live or provide a local CSV."
        )
    position_report_path, reference_table = rebuild_position_report_from_existing_live_csv(
        existing_csv_path=EXISTING_LIVE_CSV,
        output_path=POSITION_REPORT_OUTPUT,
    )
    acquisition_note = "Rebuilt offline from an existing live IBKR report CSV through the current normalization stack."

position_report_df = pd.read_csv(position_report_path)
risk_input_rows = load_position_rows(position_report_path, security_reference_table=reference_table)
risk_input_df = pd.DataFrame([row.__dict__ for row in risk_input_rows])

print(acquisition_note)
print(f"Position report path: {position_report_path}")
print(f"Position rows: {len(position_report_df)}")
mapped_rows = int((risk_input_df["mapping_status"] == "mapped").sum())
print(f"Mapped rows: {mapped_rows} / {len(risk_input_df)}")

preview_columns = [
    "account",
    "internal_id",
    "symbol",
    "display_ticker",
    "display_name",
    "asset_class",
    "quantity",
    "market_value",
    "signed_exposure_usd",
    "eq_country",
    "eq_sector",
    "fi_tenor",
    "mapping_status",
]
display(
    risk_input_df.loc[:, preview_columns]
    .sort_values("market_value", ascending=False, na_position="last")
    .reset_index(drop=True)
)


In [ ]:
funded_aum = _funded_aum(risk_input_rows)
asset_class_summary = (
    risk_input_df.groupby("asset_class", as_index=False)
    .agg(
        positions=("internal_id", "count"),
        net_exposure_usd=("signed_exposure_usd", "sum"),
        gross_exposure_usd=("gross_exposure_usd", "sum"),
        market_value_usd=("market_value", "sum"),
    )
    .sort_values("gross_exposure_usd", ascending=False)
    .reset_index(drop=True)
)
asset_class_summary["gross_pct_of_aum"] = asset_class_summary["gross_exposure_usd"] / funded_aum

top_positions = (
    risk_input_df.loc[:, [
        "display_ticker",
        "display_name",
        "asset_class",
        "quantity",
        "market_value",
        "signed_exposure_usd",
        "eq_country",
        "eq_sector",
        "fi_tenor",
    ]]
    .sort_values("market_value", ascending=False, key=lambda col: col.abs())
    .head(15)
    .reset_index(drop=True)
)

zero_loadings = {row.internal_id: 0.0 for row in risk_input_rows}
country_breakdown = breakdown_df(_build_eq_country_breakdown(risk_input_rows, zero_loadings))
sector_breakdown = breakdown_df(_build_us_sector_breakdown(risk_input_rows, zero_loadings))
fi_tenor_breakdown = breakdown_df(_build_fi_tenor_breakdown(risk_input_rows, zero_loadings))

portfolio_overview = pd.DataFrame(
    [
        {"metric": "Funded AUM", "value": funded_aum},
        {"metric": "Net exposure", "value": float(risk_input_df["signed_exposure_usd"].sum())},
        {"metric": "Gross exposure", "value": float(risk_input_df["gross_exposure_usd"].sum())},
        {"metric": "Mapped rows", "value": int((risk_input_df["mapping_status"] == "mapped").sum())},
        {"metric": "Total rows", "value": int(len(risk_input_df))},
    ]
)

display(Markdown("## Current Portfolio Report Pieces"))
display(portfolio_overview)

display(Markdown("### Top Positions"))
display(top_positions)

display(Markdown("### Asset Class Summary"))
display(asset_class_summary)

display(Markdown("### EQ Country Breakdown"))
display(country_breakdown)

display(Markdown("### US Sector Breakdown"))
display(sector_breakdown if not sector_breakdown.empty else pd.DataFrame([{"bucket": "No data", "parent": "US_EQ"}]))

display(Markdown("### FI Tenor Breakdown"))
display(fi_tenor_breakdown)


In [ ]:
risk_html_path = None
if TRY_BUILD_RISK_HTML:
    try:
        risk_html_path = generate_risk_html_report(
            positions_csv_path=position_report_path,
            output_path=RISK_HTML_OUTPUT,
            returns_path=Path(RETURNS_JSON) if RETURNS_JSON else None,
            proxy_path=Path(PROXY_JSON) if PROXY_JSON else None,
            regime_path=Path(REGIME_JSON) if REGIME_JSON else None,
            security_reference_path=SECURITY_REFERENCE_OUTPUT,
        )
        print(f"Generated HTML risk report: {risk_html_path}")
    except Exception as exc:
        print("Risk HTML generation is optional in this notebook and failed cleanly:")
        print(type(exc).__name__, exc)
else:
    print("TRY_BUILD_RISK_HTML is False. Set it to True if you want to attempt the HTML risk report build.")


In [ ]:
target_workbook_path = project_root / "outputs" / "reports" / "target_report.xlsx"
position_sheet_preview = preview_target_workbook_sheet(target_workbook_path, "Position")
risk_sheet_preview = preview_target_workbook_sheet(target_workbook_path, "Risk")

gap_review = pd.DataFrame(
    [
        {
            "layer": "Universe-driven security semantics",
            "status": "done",
            "what_you_can_see_now": "Stable IDs, generated security reference, and semantic fields come from security_universe.csv.",
            "main_gap": "Add local override workflow and broader rule coverage.",
        },
        {
            "layer": "Normalized current position report",
            "status": "done",
            "what_you_can_see_now": "Current notebook can build a normalized report from live TWS or from an existing live CSV.",
            "main_gap": "Improve live ergonomics and fixture coverage.",
        },
        {
            "layer": "Exposure bucket rollups",
            "status": "mostly done",
            "what_you_can_see_now": "Asset class, EQ country, US sector, and FI tenor sections are available today.",
            "main_gap": "Move from preview tables to final workbook layout.",
        },
        {
            "layer": "Risk HTML",
            "status": "partially done",
            "what_you_can_see_now": "Optional HTML report can be generated when returns/proxy data are available.",
            "main_gap": "Cache returns/proxies and add deeper attribution.",
        },
        {
            "layer": "Target workbook Position sheet parity",
            "status": "not done",
            "what_you_can_see_now": "We have the core data but not the final workbook sections/layout.",
            "main_gap": "Workbook renderer and target-style subtotal sections.",
        },
        {
            "layer": "Target workbook Risk sheet parity",
            "status": "not done",
            "what_you_can_see_now": "We have partial risk analytics and regime integration, but not target parity.",
            "main_gap": "Covariance attribution, scenario panels, and workbook formatting.",
        },
    ]
)

display(Markdown("## Target Workbook Preview"))
display(Markdown("### Position sheet (top rows)"))
display(position_sheet_preview)

display(Markdown("### Risk sheet (top rows)"))
display(risk_sheet_preview)

display(Markdown("## Gap Review"))
display(gap_review)

display(Markdown("**Practical read:** useful internal portfolio report is close; target workbook parity is still a later-stage formatting + attribution project."))
